# Nemotron 3 Nano - RAFT / Reinforce-Rej (RTX 6000 Pro, Kaggle compute-safe)

**RAFT** (Reward rAnked FineTuning, Dong 2023, arXiv:2304.06767) /
**Reinforce-Rej** (Minimalist Approach, Xiong 2025, arXiv:2504.09118).

Loop:
1. **Sample** N rollouts per prompt at T=1.0 from the current policy.
2. **Verify** each with the deterministic per-type verifier (ported from the GRPO notebook).
3. **Filter** by group pass-rate `p`:
   - `raft`         -> keep top-k correct completions; drop prompts with 0 correct.
   - `reinforce_rej`-> additionally drop **all-correct** (p=1) groups (no learning signal).
     This is Xiong 2025's finding: GRPO's edge is implicit filtering of zero-gradient
     prompts, not reward normalization.
4. **SFT** on the kept (prompt, correct-completion) pairs = self-distillation (STaR / ReST-EM).

Why RAFT here: it sidesteps the entire fragile RL-loss class that kept crashing on
Nemotron-H (Unsloth fused GRPO/SFT loss double-projects lm_head). Generate + verify
+ plain SFT only. Greedy-eval friendly: trains the model to be confidently correct.

**Compute-safety (96 GB GPU, ~12 h wall, locked weekly quota):**
- `SMOKE_TEST` first (few prompts, few steps) before burning quota.
- Sampling + SFT scopes bounded by knobs; generation memory toggled safely.
- Sampled rollouts cached to disk -> a crash in SFT doesn't lose the expensive sampling.
- `save_steps` + auto-resume on the SFT trainer.


In [ ]:
import os, sys
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="strict")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="strict")

TRAIN_ON_KAGGLE = 1
USE_PRETRAINED  = 0
assert (TRAIN_ON_KAGGLE + USE_PRETRAINED) == 1, \
    "Set exactly one of TRAIN_ON_KAGGLE / USE_PRETRAINED to 1."

BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
print({"TRAIN_ON_KAGGLE": TRAIN_ON_KAGGLE, "USE_PRETRAINED": USE_PRETRAINED})

In [ ]:
import os, glob, sys, subprocess, site

candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
print("Found Triton wheels:", candidates)
if not candidates:
    raise FileNotFoundError("No Triton wheel found under /kaggle/input")
wheel = candidates[0]

target = "/kaggle/working/pydeps"
os.makedirs(target, exist_ok=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-deps", "--target", target,
     "--upgrade", "--ignore-installed", wheel],
    check=True,
)
if target not in sys.path:
    sys.path.insert(0, target)
site.addsitedir(target)

import importlib.util
print("triton spec:", importlib.util.find_spec("triton"))

In [ ]:
if TRAIN_ON_KAGGLE:
    import sys, os, shutil, stat

    sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')

    ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
    ptxas_dst = '/tmp/ptxas-blackwell'
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        src_bin = os.path.dirname(ptxas_src)
        dst_bin = '/tmp/triton_nvidia_bin'
        shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
        for f in os.listdir(dst_bin):
            fp = os.path.join(dst_bin, f)
            if os.path.isfile(fp):
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst

        import triton.backends.nvidia as nv_backend
        nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
        os.environ['TRITON_PTXAS_PATH'] = ptxas_dst

    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: '12.0'
    print('Training environment fixes applied.')
else:
    print("USE_PRETRAINED=1: skipping Triton / ptxas environment fixes.")

In [ ]:
import os

BASE_MODEL_NAME  = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
RAFT_DATA_PATH   = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"
# *** STRONGLY recommended: warm-start from a prior SFT adapter. Cold-start LoRA
#     usually solves 0/N rollouts on these puzzles -> empty corpus.
SFT_ADAPTER_DIR  = "/kaggle/input/your-sft-adapter"   # edit to a real path

OUTPUT_ROOT      = "outputs"
ROLLOUT_CACHE    = os.path.join(OUTPUT_ROOT, "raft_rollouts.jsonl")
RAFT_ADAPTER_DIR = os.path.join(OUTPUT_ROOT, "raft_adapter")
SUBMISSION_DIR   = os.path.join(OUTPUT_ROOT, "submission_raft")
TB_LOG_DIR       = os.path.join(OUTPUT_ROOT, "tb_logs_raft")
os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(TB_LOG_DIR, exist_ok=True)

SEED = 42

# ===========================================================================
# COMPUTE-SAFETY KNOBS  (Kaggle: 96 GB GPU, ~12 h wall, locked weekly quota)
# ===========================================================================
SMOKE_TEST   = 1

# --- algorithm ---
RAFT_MODE    = "reinforce_rej"   # "raft" | "reinforce_rej"
TOP_K_KEEP   = 1
# KEEP_FORMAT_VALID gates rollouts that lack <think>...</think> BEFORE \boxed{}.
# Default OFF -- cold-start adapters rarely emit the strict format, gate -> 0 corpus.
# Turn ON after a strong warm-start adapter is in place.
KEEP_FORMAT_VALID = False

# --- SAMPLING scope (the expensive part) ---
N_PROMPTS    = 400               # prompts from train.csv (real run)
N_ROLLOUTS   = 4                 # rollouts per prompt
GEN_MAX_NEW  = 1024              # max new tokens per rollout
GEN_TEMP     = 1.0
GEN_TOP_P    = 1.0

# *** BATCHED GENERATION: process this many prompts per model.generate call.
# Tune up if PEAK VRAM has headroom; tune down on OOM. 4 = good default at
# GEN_MAX_NEW=1024 with 96 GB. Speedup over batch=1 is ~3-5x.
BATCH_PROMPTS = 4

# --- SFT scope ---
NUM_EPOCHS    = 2
MODEL_MAX_LEN = 8192
TRAIN_MAX_LEN = 4096

# --- smoke overrides ---
SMOKE_PROMPTS = 8                # >= BATCH_PROMPTS so batching actually runs
SMOKE_ROLLOUTS = 2
SMOKE_GEN_MAX_NEW = 256
SMOKE_STEPS = 8

PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

if SMOKE_TEST:
    N_PROMPTS, N_ROLLOUTS, GEN_MAX_NEW = SMOKE_PROMPTS, SMOKE_ROLLOUTS, SMOKE_GEN_MAX_NEW

print({"SMOKE_TEST": SMOKE_TEST, "RAFT_MODE": RAFT_MODE, "N_PROMPTS": N_PROMPTS,
       "N_ROLLOUTS": N_ROLLOUTS, "GEN_MAX_NEW": GEN_MAX_NEW, "BATCH_PROMPTS": BATCH_PROMPTS,
       "KEEP_FORMAT_VALID": KEEP_FORMAT_VALID,
       "TOP_K_KEEP": TOP_K_KEEP, "NUM_EPOCHS": NUM_EPOCHS, "TRAIN_MAX_LEN": TRAIN_MAX_LEN})

In [ ]:
if TRAIN_ON_KAGGLE:
    import glob, os, subprocess, sys

    def recursive_wheels(pattern: str):
        return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

    packages_dir = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
    all_mamba  = recursive_wheels("mamba_ssm-*.whl")
    all_causal = recursive_wheels("causal*conv1d*.whl")

    import torch
    print("Torch:", torch.__version__, "CUDA:", torch.cuda.is_available())
    if not torch.cuda.is_available():
        raise RuntimeError("TRAIN_ON_KAGGLE=1 requires a GPU runtime.")
    if not os.path.isdir(packages_dir):
        raise FileNotFoundError(f"Offline wheel directory not found: {packages_dir}")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-index",
         "--find-links", packages_dir, "unsloth", "trl", "peft", "transformers",
         "datasets", "accelerate", "bitsandbytes"],
        check=True,
    )

    def pick_last(w): return w[-1] if w else None
    causal_wheel = pick_last(all_causal)
    mamba_wheel  = pick_last(all_mamba)
    if causal_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", causal_wheel], check=True)
    if mamba_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", mamba_wheel], check=True)
    else:
        raise FileNotFoundError("No compatible mamba_ssm wheel found under /kaggle/input.")
    print("Offline package installation finished.")

In [ ]:
if TRAIN_ON_KAGGLE:
    import torch
    import kagglehub
    from unsloth import FastLanguageModel

    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    print(f"Model path: {MODEL_PATH}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH,
        max_seq_length=MODEL_MAX_LEN,
        load_in_4bit=False,
        load_in_8bit=False,
        full_finetuning=False,
        trust_remote_code=True,
        unsloth_force_compile=False,
        attn_implementation="eager",
        dtype=torch.bfloat16,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    # left padding for batched GENERATION; SFT trainer cell flips to right.
    tokenizer.padding_side = "left"
    print("Model loaded with Unsloth.")
else:
    print("USE_PRETRAINED=1: skipping base model and tokenizer loading.")

## LoRA: warm-start from SFT adapter if present, else cold-start (RSLoRA r=32)

In [ ]:
from peft import PeftModel, LoraConfig, get_peft_model, TaskType
import re, os

# ---------------------------------------------------------------------------
# CRITICAL: target_modules MUST hit:
#   - self_attn.{q,k,v,o}_proj          (6 GQA attention layers, sensitive)
#   - mamba.{in,out,x,dt,gate}_proj     (pre-attention Mamba layers, sensitive)
#   - shared_experts.{gate,up,down}_proj (2 always-active experts/layer)
# and MUST NOT hit:
#   - the 128 ROUTABLE experts/layer    (sparse 6/128 -> wasted LoRA budget)
#   - lm_head                            (untied embed -> destabilizes)
#   - generic Mamba layers               (low ROI per quantization study)
#
# Past bug: `shared_expert` (singular) matched 0 -> suffix fallback attached
# LoRA to all 128 routable experts. Now: correct plural, raise on 0 matches,
# audit asserts trainable_params is in a plausible range.
# ---------------------------------------------------------------------------
linear_modules = []
for name, mod in model.named_modules():
    if mod.__class__.__name__ in ("Linear", "Linear4bit", "Linear8bitLt"):
        linear_modules.append(name)

target_regex = (
    r".*("
    r"self_attn\.(q|k|v|o)_proj"
    r"|mamba\.(in|out|x|dt|gate)_proj"
    r"|shared_experts\.(gate|up|down)_proj"
    r")$"
)
matched = [n for n in linear_modules if re.match(target_regex, n)]
print(f"LoRA target regex matched {len(matched)} modules.")
if len(matched) == 0:
    sample = [n for n in linear_modules if "expert" in n or "mamba" in n or "self_attn" in n][:20]
    raise RuntimeError(
        "LoRA target_regex matched 0 modules. NO silent fallback (would attach "
        "LoRA to all routable experts and waste budget). "
        f"Sample module names for debugging: {sample}"
    )

# ---------------------------------------------------------------------------
# Branch A: WARM-START -- if SFT_ADAPTER_DIR points at a valid adapter, load
# it on top of the base NVIDIA model (continue training).
# Branch B: COLD-START -- if no adapter found at SFT_ADAPTER_DIR, attach a
# fresh r=32 LoRA on the BASE NVIDIA checkpoint already loaded in the model-load
# cell. RAFT then trains from scratch on top of vanilla Nemotron-3-Nano-30B.
# ---------------------------------------------------------------------------
_adapter_cfg = os.path.join(SFT_ADAPTER_DIR, "adapter_config.json")
if os.path.isdir(SFT_ADAPTER_DIR) and os.path.exists(_adapter_cfg):
    print(f"[branch A: WARM-START] loading SFT adapter from {SFT_ADAPTER_DIR}")
    model = PeftModel.from_pretrained(model, SFT_ADAPTER_DIR, is_trainable=True)
    model.gradient_checkpointing_enable()
    model.print_trainable_parameters()
else:
    print(f"[branch B: COLD-START] no adapter at {SFT_ADAPTER_DIR!r}.")
    print(f"[branch B: COLD-START] starting from BASE NVIDIA checkpoint "
          f"({BASE_MODEL_NAME}) with fresh RSLoRA+DoRA r=32.")
    print(f"[branch B: COLD-START] RAFT bootstraps poorly from cold base on "
          f"this benchmark (~<20% hit rate). Expect a thin first-round corpus; "
          f"consider warm-starting from any prior SFT adapter for stronger results.")
    lora_config = LoraConfig(
        r=32, lora_alpha=64, lora_dropout=0.0, bias="none",
        target_modules=target_regex,
        task_type=TaskType.CAUSAL_LM,
        use_rslora=True,
        use_dora=True,    # Weight-Decomposed LoRA -- +2-4pp typical on reasoning
    )
    model = get_peft_model(model, lora_config)
    model.gradient_checkpointing_enable()
    model.print_trainable_parameters()

# --- audit: catches "regex matched 0 -> attached to wrong modules" failure mode
trainable = [(n, p.numel()) for n, p in model.named_parameters() if p.requires_grad]
n_train = sum(s for _, s in trainable)
print(f"[audit] {len(trainable)} trainable param tensors  total={n_train/1e6:.1f}M")
print(f"[audit] first 5: {[n for n,_ in trainable[:5]]}")
print(f"[audit] last  5: {[n for n,_ in trainable[-5:]]}")
assert 50_000_000 <= n_train <= 400_000_000, (
    f"[audit] trainable param count {n_train/1e6:.1f}M outside [50M, 400M] -- "
    "either regex missed everything (too low) or attached to routable experts "
    "(too high). Inspect target_regex against linear_modules above."
)

## Verifiable Reward (deterministic, per-type)

Ported verbatim from the GRPO notebook: classify puzzle type, brace-balanced
`\boxed{}` extraction, type-aware comparison (numeric tolerance for gravity/units,
multi-base integer match for bit/base, case-insensitive for cipher).


In [ ]:
import re

_PUZZLE_PATTERNS = [
    ("Number Base Conversion", re.compile(r"numeral system|base[- ]?\d|number.*convert|radix|secret number", re.IGNORECASE)),
    ("Gravitational Constant", re.compile(r"gravit|gravity|falling|free.?fall|acceleration due to", re.IGNORECASE)),
    ("Equation Transformation", re.compile(r"transformation rule|equation.*transform|secret.*rule.*equation|rule.*applied.*equation", re.IGNORECASE)),
    ("Text Encryption",         re.compile(r"encrypt|cipher|secret.*code.*letter|coded.*message|secret.*text", re.IGNORECASE)),
    ("Bit Manipulation",        re.compile(r"bit.?manipul|binary|8.?bit|bitwise|bit.*transform", re.IGNORECASE)),
    ("Unit Conversion",         re.compile(r"unit.?conver|measurement|becomes.*\d|secret.*conver.*measur", re.IGNORECASE)),
]

def classify_puzzle(prompt: str) -> str:
    for label, pat in _PUZZLE_PATTERNS:
        if pat.search(prompt or ""):
            return label
    return "Unknown"

_BOXED_RE = re.compile(r"\\boxed\{([^}]*)\}")
_THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)

def extract_boxed(text):
    idx = text.find("\\boxed{")
    if idx == -1:
        m = _BOXED_RE.search(text)
        return m.group(1).strip() if m else None
    depth, start = 1, idx + 7
    for i in range(start, len(text)):
        if text[i] == "{":   depth += 1
        elif text[i] == "}": depth -= 1
        if depth == 0:       return text[start:i].strip()
    return text[start:].strip()

def _norm(s):
    return str(s).strip().lower().replace(" ", "").replace(",", "")

def _numeric_match(predicted, expected, rel_tol=1e-2, abs_tol=1e-4):
    try:
        pf = float(str(predicted).replace(",", "").strip())
        ef = float(str(expected).replace(",", "").strip())
    except (ValueError, TypeError):
        return False
    return abs(pf - ef) <= max(rel_tol * max(1.0, abs(ef)), abs_tol)

def _integer_match(predicted, expected, bases=(2,8,10,16)):
    p_str, e_str = _norm(predicted), _norm(expected)
    def parse(s, base):
        try:
            if base == 16 and s.startswith("0x"): s = s[2:]
            elif base == 2 and s.startswith("0b"): s = s[2:]
            return int(s, base)
        except ValueError:
            return None
    for b1 in bases:
        pv = parse(p_str, b1)
        if pv is None: continue
        for b2 in bases:
            ev = parse(e_str, b2)
            if ev is not None and pv == ev:
                return True
    return False

def verify_answer(predicted, expected, puzzle_type):
    if predicted is None:
        return False
    pn, en = _norm(predicted), _norm(expected)
    if pn == en:
        return True
    if puzzle_type in ("Gravitational Constant", "Unit Conversion"):
        return _numeric_match(predicted, expected, rel_tol=1e-2, abs_tol=1e-4)
    if puzzle_type in ("Bit Manipulation", "Number Base Conversion"):
        return _integer_match(predicted, expected)
    if puzzle_type == "Text Encryption":
        return pn.replace("'", "") == en.replace("'", "")
    if puzzle_type == "Equation Transformation":
        return _numeric_match(predicted, expected) or pn == en
    return _numeric_match(predicted, expected)

def has_think_before_boxed(text):
    has_box = bool(_BOXED_RE.search(text)) or ("\\boxed{" in text)
    has_think = bool(_THINK_RE.search(text))
    return has_think and has_box and (text.find("</think>") < text.rfind("\\boxed{"))

# smoke
assert verify_answer("42", "42", "Unknown")
assert verify_answer("9.81", "9.80", "Gravitational Constant")
assert verify_answer("0xFF", "255", "Number Base Conversion")
assert verify_answer("HELLO", "hello", "Text Encryption")
print("Verifier smoke tests pass.")

## Generalized System Prompt (identical at sample / train / eval)

In [ ]:
SYSTEM_PROMPT = """You are a meticulous reasoning engine for a logical-puzzle benchmark. \
Every task has exactly one correct, deterministic answer that can be derived by \
exact rule-following and arithmetic. Speed does not matter; correctness does.

GENERAL METHOD
1. Read the problem twice. Identify the category and restate, in your own words, \
the exact transformation or quantity being asked for.
2. Extract every given value, rule, mapping, base, unit, and constant verbatim. \
Never invent data that is not stated.
3. Work strictly step by step. Show each intermediate result. Do arithmetic \
digit by digit and re-check it. When a rule is defined in the prompt, apply it \
literally rather than relying on prior assumptions.
4. Verify: substitute your answer back into the problem and confirm it satisfies \
every stated condition. If it does not, find your error and redo the step.

CATEGORY-SPECIFIC RULES
- Bit manipulation: operate on the stated bit width (default 8 bits). Preserve \
leading zeros. Apply AND/OR/XOR/NOT/shifts exactly; for shifts state whether bits \
fall off or wrap as specified. Report the result in the format the prompt uses \
(binary string, or decimal if asked).
- Number-base conversion: convert through base 10 as an intermediate when helpful. \
Map digits A-F (and beyond) carefully. State the source and target base. Do not \
add base prefixes unless asked.
- Gravitational constant / free-fall: use the constant exactly as given in the \
prompt (do not substitute a textbook g). Track units through every step. Round \
only at the end, to the precision the prompt implies.
- Unit conversion: write the conversion factor as an explicit fraction, cancel \
units, and keep full precision until the final rounding. State the final unit.
- Text encryption / cipher: determine the exact scheme (shift/Caesar, substitution, \
keyword, etc.) and direction (encrypt vs decrypt). Transform one character at a \
time, preserving case, spacing, and punctuation unless told otherwise.
- Algebraic equations & equation transformation: isolate the target symbol with \
inverse operations applied to both sides; or, if a transformation rule is defined, \
apply that exact rule. Keep equations balanced at every line.

OUTPUT CONTRACT (mandatory)
- First think inside a single <think> ... </think> block containing your full \
step-by-step derivation and verification.
- Immediately after </think>, output the final answer once, wrapped exactly as \
\\boxed{...} with nothing after it.
- The boxed content must be only the answer in the form the problem expects \
(e.g. an 8-bit binary string, a number, a word, or an expression) - no units \
unless the problem asks for them, no extra words."""

print(f"SYSTEM_PROMPT chars: {len(SYSTEM_PROMPT)}")

## Load prompts for sampling (raw competition train.csv)

In [ ]:
import pandas as pd

df = pd.read_csv(RAFT_DATA_PATH)
print(f"Data: {len(df)} rows.  Columns: {list(df.columns)}")
if not {"prompt", "answer"}.issubset(df.columns):
    raise ValueError(f"train.csv must have 'prompt' and 'answer'; got {list(df.columns)}")

df = df.dropna(subset=["prompt", "answer"]).reset_index(drop=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
df = df.head(N_PROMPTS).reset_index(drop=True)

prompts = []
for _, row in df.iterrows():
    prompts.append({"raw_prompt": str(row["prompt"]), "answer": str(row["answer"])})

# type distribution of the sampling set
from collections import Counter
dist = Counter(classify_puzzle(p["raw_prompt"]) for p in prompts)
print(f"Sampling {len(prompts)} prompts. Type dist: {dict(dist)}")

## Sample N rollouts per prompt, verify, cache to disk

Generation memory is toggled safely (gradient checkpointing OFF + KV cache ON for
gen, restored after). Rollouts + correctness are written to `ROLLOUT_CACHE` so a
later SFT crash never costs the expensive sampling. If the cache already exists it
is reused (delete it to re-sample).


In [ ]:
import json, time, gc, torch

def build_gen_prompt(raw_prompt):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": raw_prompt + PROMPT_SUFFIX}]
    try:
        return tokenizer.apply_chat_template(msgs, tokenize=False,
                                             add_generation_prompt=True, enable_thinking=True)
    except TypeError:
        return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def _set_gen_mode(on):
    try:
        if on: model.gradient_checkpointing_disable()
        else:  model.gradient_checkpointing_enable()
    except Exception as e:
        print("grad-ckpt toggle warn:", e)
    try:
        model.config.use_cache = on
    except Exception:
        try: model.base_model.config.use_cache = on
        except Exception: pass

if os.path.exists(ROLLOUT_CACHE) and not SMOKE_TEST:
    print(f"Rollout cache exists -> reusing {ROLLOUT_CACHE} (delete to re-sample).")
else:
    if os.path.exists(ROLLOUT_CACHE):
        os.remove(ROLLOUT_CACHE)
    _set_gen_mode(True)
    model.eval()
    tokenizer.padding_side = "left"     # required for batched gen

    torch.cuda.empty_cache(); gc.collect(); torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    n_done = n_correct_rollouts = 0

    fout = open(ROLLOUT_CACHE, "w", encoding="utf-8")
    try:
        for batch_start in range(0, len(prompts), BATCH_PROMPTS):
            batch = prompts[batch_start:batch_start + BATCH_PROMPTS]
            texts = [build_gen_prompt(ex["raw_prompt"]) for ex in batch]

            enc = tokenizer(
                texts, return_tensors="pt", padding=True, truncation=True,
                max_length=TRAIN_MAX_LEN,
            ).to(model.device)
            plen = enc["input_ids"].shape[1]    # left-padded -> all prompts end at column plen

            with torch.no_grad():
                out = model.generate(
                    **enc,
                    max_new_tokens=GEN_MAX_NEW,
                    do_sample=True,
                    temperature=GEN_TEMP,
                    top_p=GEN_TOP_P,
                    num_return_sequences=N_ROLLOUTS,
                    pad_token_id=tokenizer.pad_token_id,
                )
            # out shape: (len(batch) * N_ROLLOUTS, plen + gen_len)
            # left padding means generation starts at column `plen` for every row.

            for i, ex in enumerate(batch):
                seqs = out[i * N_ROLLOUTS : (i + 1) * N_ROLLOUTS]
                ptype = classify_puzzle(ex["raw_prompt"])
                comps, flags = [], []
                for seq in seqs:
                    gen = tokenizer.decode(seq[plen:], skip_special_tokens=True)
                    pred = extract_boxed(gen)
                    ok = bool(verify_answer(pred, ex["answer"], ptype))
                    comps.append(gen)
                    flags.append(ok)
                    n_correct_rollouts += int(ok)
                fout.write(json.dumps({
                    "raw_prompt": ex["raw_prompt"], "answer": ex["answer"],
                    "ptype": ptype, "completions": comps, "correct": flags,
                }) + "\n")
                n_done += 1

            del out, enc
            torch.cuda.empty_cache()
            el = time.time() - t0
            rate = el / max(1, n_done)
            print(f"  [{n_done}/{len(prompts)}] {el/60:.1f} min  "
                  f"~{rate:.1f}s/prompt  corr_rollouts={n_correct_rollouts}  "
                  f"peakVRAM={torch.cuda.max_memory_allocated()/1e9:.1f}GB")
    finally:
        fout.close()
        _set_gen_mode(False)
        model.train()

    el = time.time() - t0
    print(f"\nSampling done: {n_done} prompts x {N_ROLLOUTS} in {el/60:.1f} min")
    print(f"EST for full set: {el/max(1,n_done):.1f}s/prompt -> "
          f"9500 prompts ~= {9500*el/max(1,n_done)/3600:.1f} h at these knobs")
    print(f"Hit rate: {n_correct_rollouts}/{n_done*N_ROLLOUTS} rollouts correct "
          f"({100*n_correct_rollouts/max(1,n_done*N_ROLLOUTS):.1f}%)")
    print(f"PEAK VRAM (gen): {torch.cuda.max_memory_allocated()/1e9:.1f} GB")
    if n_correct_rollouts == 0:
        print("\n[WARN] 0 correct rollouts. Cold-start cannot bootstrap RAFT on this benchmark. "
              "Either (a) warm-start from an SFT adapter (set SFT_ADAPTER_DIR), or "
              "(b) raise N_ROLLOUTS to 8+ and try again, or "
              "(c) use easier prompts (filter by p_type or seed).")

## Filter rollouts -> SFT corpus (RAFT / Reinforce-Rej)

Per prompt, pass-rate `p = #correct / N`. `raft` keeps top-k correct completions
from any prompt with >=1 correct. `reinforce_rej` additionally drops `p==1`
(all-correct) prompts -- those carry no learning signal (Xiong 2025). Kept
completions become the assistant targets (self-distillation).


In [ ]:
import json
from datasets import Dataset as HFDataset

groups = []
with open(ROLLOUT_CACHE, encoding="utf-8") as f:
    for line in f:
        groups.append(json.loads(line))


def build_corpus(mode, keep_format, allow_any_completion=False):
    """Apply RAFT/Reinforce-Rej filtering.

    mode               : 'raft' keeps any group with >=1 correct.
                         'reinforce_rej' additionally drops all-correct (p=1) groups.
    keep_format        : require <think>...</think> BEFORE \boxed{} on the kept completion.
    allow_any_completion: LAST-RESORT mode -- if no correct rollouts exist anywhere,
                         keep the shortest completion regardless of correctness so
                         the trainer at least sees format examples (NOT used by default).
    """
    n_all_wrong = n_all_correct = n_mixed = 0
    out = []
    for g in groups:
        flags = g["correct"]
        n = len(flags); n_ok = sum(flags)
        p = n_ok / max(1, n)
        if n_ok == 0:
            n_all_wrong += 1
            if not allow_any_completion:
                continue
            cand = list(g["completions"])    # last-resort: keep any
        else:
            if p >= 1.0:
                n_all_correct += 1
                if mode == "reinforce_rej":
                    continue
            else:
                n_mixed += 1
            cand = [c for c, ok in zip(g["completions"], flags) if ok]

        if keep_format:
            fmt = [c for c in cand if has_think_before_boxed(c)]
            cand = fmt if fmt else cand
        cand = sorted(cand, key=len)[:TOP_K_KEEP]   # prefer shorter (cleaner) CoT

        for comp in cand:
            out.append({
                "system": SYSTEM_PROMPT,
                "user":   g["raw_prompt"] + PROMPT_SUFFIX,
                "assistant": comp.strip(),
            })
    return out, dict(all_wrong=n_all_wrong, all_correct=n_all_correct, mixed=n_mixed)


# Try strict mode first
records, stats = build_corpus(RAFT_MODE, KEEP_FORMAT_VALID)
print(f"[try1] mode={RAFT_MODE} format={KEEP_FORMAT_VALID} -> "
      f"corpus={len(records)} groups={stats}")

# Fallback 1: drop format gate
if len(records) == 0 and KEEP_FORMAT_VALID:
    records, stats = build_corpus(RAFT_MODE, False)
    print(f"[try2] mode={RAFT_MODE} format=False -> corpus={len(records)} groups={stats}")

# Fallback 2: switch to raft (keep all-correct, signal-but-no-contrast)
if len(records) == 0 and RAFT_MODE == "reinforce_rej":
    records, stats = build_corpus("raft", False)
    print(f"[try3] mode=raft format=False -> corpus={len(records)} groups={stats}")

# Hard stop if still nothing -- means 0 correct rollouts anywhere.
if len(records) == 0:
    raise RuntimeError(
        "Empty corpus after all fallbacks: 0 correct rollouts in cache. "
        "Cold-start RAFT cannot bootstrap on this benchmark. Fixes: "
        "(a) WARM-START -- set SFT_ADAPTER_DIR to a real prior SFT adapter, then re-run; "
        "(b) raise N_ROLLOUTS to 8+ to widen the search; "
        "(c) seed easier prompts first (e.g. only bit-manipulation). "
        "Delete `outputs/raft_rollouts.jsonl` before re-sampling."
    )

raw_ds = HFDataset.from_list(records)
print(f"\nFinal corpus: {len(records)} examples")
print("--- sample assistant target ---\n", records[0]["assistant"][:500])

In [ ]:
def tokenize_with_assistant_mask(example):
    full_msgs = [
        {"role": "system",    "content": example["system"]},
        {"role": "user",      "content": example["user"]},
        {"role": "assistant", "content": example["assistant"]},
    ]
    prefix_msgs = [
        {"role": "system", "content": example["system"]},
        {"role": "user",   "content": example["user"]},
    ]

    def render(msgs, add_gen):
        try:
            return tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=add_gen, enable_thinking=True)
        except TypeError:
            return tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=add_gen)

    full_text   = render(full_msgs, False)
    prefix_text = render(prefix_msgs, True)

    full_ids   = tokenizer(full_text, add_special_tokens=False, truncation=True,
                           max_length=TRAIN_MAX_LEN)["input_ids"]
    prefix_ids = tokenizer(prefix_text, add_special_tokens=False)["input_ids"]

    cutoff = min(len(prefix_ids), len(full_ids))
    labels = list(full_ids)
    for i in range(cutoff):
        labels[i] = -100
    return {"input_ids": full_ids, "labels": labels}


tokenized_ds = raw_ds.map(tokenize_with_assistant_mask,
                          remove_columns=raw_ds.column_names,
                          desc="Tokenize + assistant mask")

def _has_label(ex):
    return any(t != -100 for t in ex["labels"])
before = len(tokenized_ds)
tokenized_ds = tokenized_ds.filter(_has_label)
print(f"Kept {len(tokenized_ds)}/{before} rows with >=1 unmasked assistant token.")

import numpy as np
_lens = np.array([len(x) for x in tokenized_ds["input_ids"]])
pct = lambda p: int(np.percentile(_lens, p))
print(f"Token length  min={_lens.min()}  mean={_lens.mean():.0f}  "
      f"p50={pct(50)}  p90={pct(90)}  p99={pct(99)}  max={_lens.max()}")

In [ ]:
import torch

class CompletionOnlyDataCollator:
    """Pads input_ids/labels/attention_mask; preserves -100 on prompt tokens."""
    def __init__(self, tokenizer, label_pad_id=-100):
        self.pad_id = tokenizer.pad_token_id
        self.label_pad_id = label_pad_id

    def __call__(self, features):
        maxlen = max(len(f["input_ids"]) for f in features)
        input_ids, labels, attn = [], [], []
        for f in features:
            ids = list(f["input_ids"]); lab = list(f["labels"])
            pad = maxlen - len(ids)
            input_ids.append(ids + [self.pad_id] * pad)
            labels.append(lab + [self.label_pad_id] * pad)
            attn.append([1] * len(ids) + [0] * pad)
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attn, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

# SFT uses RIGHT padding for causal LM loss (we set LEFT earlier for generation).
tokenizer.padding_side = "right"
data_collator = CompletionOnlyDataCollator(tokenizer)
print("Collator ready. padding_side =", tokenizer.padding_side)

## Vanilla TRL SFT + memory-safe loss

Same fix as the other notebooks: purge Unsloth (its fused loss double-projects
lm_head on Nemotron-H) and subclass vanilla `SFTTrainer` with fused
`cross_entropy` (no fp32 full-vocab log_softmax spike). RAFT trains with plain
mean NLL on the kept correct completions.


In [ ]:
import os, sys
os.environ["TORCHDYNAMO_DISABLE"]   = "1"
os.environ["TORCH_COMPILE_DISABLE"] = "1"

import torch, torch._dynamo
torch._dynamo.config.disable = True
torch._dynamo.reset()

for _m in list(sys.modules):
    if _m == "trl" or _m.startswith("trl.") or "unsloth" in _m.lower():
        del sys.modules[_m]
sys.meta_path = [f for f in sys.meta_path
                 if "unsloth" not in type(f).__module__.lower()]

import torch.nn.functional as F
from trl import SFTTrainer, SFTConfig
assert "unsloth" not in SFTTrainer.__module__.lower(), \
    f"Still using Unsloth trainer: {SFTTrainer.__module__}"
print(f"SFTTrainer module: {SFTTrainer.__module__}  (vanilla TRL, dynamo disabled)")


class SafeSFTTrainer(SFTTrainer):
    """Vanilla SFT with fused, memory-safe mean-NLL loss. `**kwargs` swallows
    num_items_in_batch from newer transformers."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        shift_logits = logits[:, :-1, :]
        shift_labels = labels[:, 1:].to(shift_logits.device)
        B, T, V = shift_logits.shape
        loss = F.cross_entropy(
            shift_logits.reshape(-1, V),
            shift_labels.reshape(-1),
            ignore_index=-100,
        )
        if not torch.isfinite(loss):
            loss = (logits.sum() * 0.0).requires_grad_(True)
        return (loss, outputs) if return_outputs else loss


print("SafeSFTTrainer defined.")

In [ ]:
_max_steps = SMOKE_STEPS if SMOKE_TEST else -1

sft_config = SFTConfig(
    output_dir                   = os.path.join(OUTPUT_ROOT, "raft_sft_run"),
    num_train_epochs             = NUM_EPOCHS,
    max_steps                    = _max_steps,
    per_device_train_batch_size  = 1,
    gradient_accumulation_steps  = 8,
    learning_rate                = 5e-5,        # slightly below SFT-from-scratch; RAFT refines
    lr_scheduler_type            = "cosine",
    warmup_ratio                 = 0.05,
    weight_decay                 = 0.01,
    max_grad_norm                = 1.0,
    optim                        = "paged_adamw_8bit",
    adam_beta1                   = 0.9,
    adam_beta2                   = 0.999,
    bf16                         = True,
    gradient_checkpointing       = True,
    gradient_checkpointing_kwargs= {"use_reentrant": True},
    max_length                   = TRAIN_MAX_LEN,
    packing                      = False,
    dataset_kwargs               = {"skip_prepare_dataset": True},
    remove_unused_columns        = False,
    logging_steps                = 1 if SMOKE_TEST else 5,
    logging_dir                  = TB_LOG_DIR,
    report_to                    = "none",
    save_strategy                = "no" if SMOKE_TEST else "steps",
    save_steps                   = 100,
    save_total_limit             = 2,
    seed                         = SEED,
    dataloader_num_workers       = 2,
)
print(f"SFTConfig ready. mode={'SMOKE' if SMOKE_TEST else 'REAL'}  "
      f"max_steps={_max_steps}  epochs={NUM_EPOCHS}  "
      f"eff_batch={sft_config.per_device_train_batch_size*sft_config.gradient_accumulation_steps}")

## Launch RAFT SFT (smoke first)

In [ ]:
import gc, time, torch, os, glob, shutil

trainer = SafeSFTTrainer(
    model           = model,
    args            = sft_config,
    train_dataset   = tokenized_ds,
    data_collator   = data_collator,
    processing_class= tokenizer,
)

_ckpts = sorted(glob.glob(os.path.join(sft_config.output_dir, "checkpoint-*")),
                key=lambda p: int(p.rsplit("-", 1)[-1]))
resume = bool(_ckpts) and not SMOKE_TEST
print(f"{'Resuming from' if resume else 'Fresh start; no'} checkpoint in {sft_config.output_dir}")

torch.cuda.empty_cache(); gc.collect()
torch.cuda.reset_peak_memory_stats()
t0 = time.time()

train_err = None
try:
    trainer.train(resume_from_checkpoint=resume)
    print(f"RAFT SFT done in {(time.time()-t0)/60:.1f} min")
except Exception as e:
    train_err = e
    print(f"[TRAIN ERROR after {(time.time()-t0)/60:.1f} min] {type(e).__name__}: {e}")
    print("[recovery] will still try to save whatever progressed so far.")

print(f"PEAK VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f} GB / "
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")


def _save_adapter(dest):
    """Save LoRA adapter robustly. Tries PEFT-aware save first, then trainer
    save, then copies from latest checkpoint dir. Verifies required files."""
    os.makedirs(dest, exist_ok=True)
    needed = ["adapter_config.json", "adapter_model.safetensors"]

    # 1) PEFT-aware: the trainer's underlying model is the PeftModel (canonical
    # adapter-save path, handles Unsloth-wrapped models too).
    try:
        inner = trainer.model
        if hasattr(inner, "save_pretrained"):
            inner.save_pretrained(dest)
        tokenizer.save_pretrained(dest)
    except Exception as e:
        print(f"[save] inner.save_pretrained failed: {e}; trying trainer.save_model")
        try:
            trainer.save_model(dest)
            tokenizer.save_pretrained(dest)
        except Exception as e2:
            print(f"[save] trainer.save_model also failed: {e2}")

    # 2) verify -> if still missing, copy from latest checkpoint dir
    missing = [n for n in needed if not os.path.exists(os.path.join(dest, n))]
    if missing:
        ckpts = sorted(glob.glob(os.path.join(sft_config.output_dir, "checkpoint-*")),
                       key=lambda p: int(p.rsplit("-", 1)[-1]))
        if ckpts:
            src = ckpts[-1]
            print(f"[save] {missing} missing in {dest}; copying from {src}")
            for fname in needed:
                sp = os.path.join(src, fname)
                if os.path.exists(sp):
                    shutil.copy2(sp, os.path.join(dest, fname))

    # 3) final verify
    have = {n: os.path.exists(os.path.join(dest, n)) for n in needed}
    sizes = {n: (os.path.getsize(os.path.join(dest, n)) / 1024 / 1024 if have[n] else 0)
             for n in needed}
    print(f"[save] {dest} -> have={have}  sizes_MB={ {k: f'{v:.1f}' for k, v in sizes.items()} }")
    return all(have.values())


# Always save -- smoke runs need an inspectable artifact too, real runs need it
# even if train() exited noisily.
ok = _save_adapter(RAFT_ADAPTER_DIR)
if not ok:
    print("[save] WARNING: required files still missing. Available checkpoint dirs:",
          glob.glob(os.path.join(sft_config.output_dir, "checkpoint-*")))
else:
    print(f"Adapter saved + verified -> {RAFT_ADAPTER_DIR}")

if SMOKE_TEST:
    print("\n[SMOKE] green if no OOM/nan + PEAK VRAM has headroom.")

if train_err is not None:
    raise train_err

## Greedy Sanity Check (confirm \\boxed{} emitted)

In [ ]:
import torch

model.eval()
try: model.config.use_cache = True
except Exception: pass
_probe = prompts[0]["raw_prompt"] + PROMPT_SUFFIX
_msgs = [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user",   "content": _probe}]
try:
    _txt = tokenizer.apply_chat_template(_msgs, tokenize=False, add_generation_prompt=True, enable_thinking=True)
except TypeError:
    _txt = tokenizer.apply_chat_template(_msgs, tokenize=False, add_generation_prompt=True)

_inputs = tokenizer(_txt, return_tensors="pt").to(model.device)
with torch.no_grad():
    _out = model.generate(**_inputs, max_new_tokens=512, do_sample=False, temperature=None, top_p=None)
_gen = tokenizer.decode(_out[0][_inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(_gen[:1200])
print("\nHAS_BOXED:", "\\boxed{" in _gen)
try: model.config.use_cache = False
except Exception: pass
model.train()

## Package submission.zip

In [ ]:
import json, shutil, zipfile, os, glob, sys, subprocess

needed = ["adapter_config.json", "adapter_model.safetensors"]
src_dir = RAFT_ADAPTER_DIR

# Kaggle eval reads submission.zip from /kaggle/working root, NOT from a subdir.
# Old packaging wrote inside OUTPUT_ROOT -> Kaggle never found it. Write to
# working-root explicitly; fall back to OUTPUT_ROOT off-Kaggle.
WORKING = "/kaggle/working" if os.path.isdir("/kaggle/working") else OUTPUT_ROOT
print(f"[package] WORKING = {WORKING}")

def _have_all(d):
    return all(os.path.exists(os.path.join(d, n)) for n in needed)

# 1) Recovery from checkpoint dir if main missing
ckpts = []
if not _have_all(src_dir):
    print(f"[package] {needed} not all present in {src_dir}")
    output_dir = sft_config.output_dir if "sft_config" in dir() else None
    ckpts = sorted(
        glob.glob(os.path.join(output_dir, "checkpoint-*")) if output_dir else [],
        key=lambda p: int(p.rsplit("-", 1)[-1]),
    )
    if ckpts:
        ck = ckpts[-1]
        print(f"[package] copying from latest checkpoint: {ck}")
        os.makedirs(src_dir, exist_ok=True)
        for fname in needed:
            sp = os.path.join(ck, fname)
            if os.path.exists(sp):
                shutil.copy2(sp, os.path.join(src_dir, fname))

    if not _have_all(src_dir) and "trainer" in dir():
        print(f"[package] still missing -- attempting trainer.model.save_pretrained({src_dir})")
        try:
            trainer.model.save_pretrained(src_dir)
            tokenizer.save_pretrained(src_dir)
        except Exception as e:
            print(f"[package] live save failed: {e}")

missing = [n for n in needed if not os.path.exists(os.path.join(src_dir, n))]
if missing:
    raise FileNotFoundError(
        f"Adapter files {missing} still missing after recovery attempts.\n"
        f"  Checked: {src_dir}\n"
        f"  Available checkpoints: {ckpts if ckpts else 'none'}\n"
        f"  Re-run the launch cell to retrain + save, or manually copy a checkpoint."
    )

# 2) Post-processing surgery: rename keys + unfuse experts + fuse Mamba (optional
# SV-amplify). Reads src_dir, writes vLLM-clean adapter to processed_dir. Output
# of this step is what we zip. Override boost via env: POSTPROC_BOOST=1.12 for
# the SV-amplification A/B test (top 50% singular values * 1.12). Default 1.0.
POSTPROC_BOOST = float(os.environ.get("POSTPROC_BOOST", "1.0"))
POSTPROC_TOP_FRAC = float(os.environ.get("POSTPROC_TOP_FRAC", "0.5"))
processed_dir = src_dir + "_processed"

script_path = None
for cand in ["tools/postprocess_adapter.py",
             "/kaggle/working/tools/postprocess_adapter.py",
             os.path.join(os.getcwd(), "tools", "postprocess_adapter.py")]:
    if os.path.exists(cand):
        script_path = cand
        break

if script_path:
    cmd = [sys.executable, script_path,
           "--in",  src_dir,
           "--out", processed_dir,
           "--boost", str(POSTPROC_BOOST),
           "--top-frac", str(POSTPROC_TOP_FRAC),
           "--rank", "32"]
    print(f"[package] running adapter post-processing: {' '.join(cmd)}")
    rc = subprocess.run(cmd, check=False).returncode
    if rc != 0 or not all(os.path.exists(os.path.join(processed_dir, n)) for n in needed):
        print(f"[package] post-processing failed (rc={rc}); falling back to raw adapter")
        processed_dir = src_dir
else:
    print(f"[package] tools/postprocess_adapter.py NOT FOUND; using raw adapter "
          f"(submission may fail at eval if module names do not match vLLM)")
    processed_dir = src_dir

# 3) Copy chosen adapter (processed or raw fallback) -> submission dir + final patch
os.makedirs(SUBMISSION_DIR, exist_ok=True)
for fname in needed:
    sp = os.path.join(processed_dir, fname)
    dp = os.path.join(SUBMISSION_DIR, fname)
    shutil.copy2(sp, dp)
    print(f"  copied {fname}  ({os.path.getsize(dp)/1024/1024:.1f} MB)")

cfg_path = os.path.join(SUBMISSION_DIR, "adapter_config.json")
with open(cfg_path) as f: cfg = json.load(f)
cfg["base_model_name_or_path"] = BASE_MODEL_NAME
cfg["inference_mode"] = True
cfg["lora_dropout"]   = 0.0
with open(cfg_path, "w") as f: json.dump(cfg, f, indent=2)

# 4) Write zip at WORKING root (Kaggle convention) so eval picks it up.
zip_path = os.path.join(WORKING, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in needed:
        zf.write(os.path.join(SUBMISSION_DIR, fname), fname)
print(f"\nsubmission.zip: {zip_path}  ({os.path.getsize(zip_path)/1024/1024:.1f} MB) - ready "
      f"(boost={POSTPROC_BOOST}  top_frac={POSTPROC_TOP_FRAC}).")

## Iterate (optional, next session)

RAFT is a loop. After the real run:
1. Upload `outputs/raft_adapter` as a Kaggle dataset, point `SFT_ADAPTER_DIR` at it.
2. Delete `raft_rollouts.jsonl` so the stronger model re-samples.
3. Re-run. Prompts that were all-wrong last round may now become mixed (curriculum
   moves up). 2-3 rounds typically.

Watch: corpus size should grow across rounds (more prompts solvable), and the
`all_wrong` count should shrink.
